# Propagation de contraintes RCC8

Ce notebook montre comment le raisonneur RCC8 propage des contraintes spatiales avec l'algorithme de path-consistency PC-2.

Objectifs:

- partir d'un reseau completement non contraint;
- ajouter des relations connues entre regions;
- utiliser la table de composition RCC8 pour reduire les domaines possibles;
- observer la propagation jusqu'au point fixe;
- voir comment une incoherence indirecte est detectee.

## 1. Initialisation

Un reseau RCC8 associe a chaque paire de regions un ensemble de relations possibles. Au depart, si on ne sait rien, chaque paire peut avoir n'importe laquelle des huit relations RCC8.

In [1]:
from rcc8.composition_table import COMPOSE
from rcc8.rcc8solver import RCC8Solver
from rcc8.relations import ALL_RELATIONS, inverse_relations

RELATION_ORDER = ["DC", "EC", "PO", "EQ", "TPP", "NTPP", "TPPI", "NTPPI"]


def build_solver(regions):
    """Cree un reseau RCC8 completement non contraint."""
    constraints = {}
    for i in regions:
        for j in regions:
            if i != j:
                constraints[(i, j)] = set(ALL_RELATIONS)
    return RCC8Solver(list(regions), constraints)


def format_relations(relations):
    """Affiche un ensemble de relations dans un ordre stable."""
    ordered = [r for r in RELATION_ORDER if r in relations]
    return "{" + ", ".join(ordered) + "}"


def print_matrix(solver, title):
    """Affiche la matrice des contraintes du reseau."""
    print(title)
    header = "      " + " | ".join(f"{v:^26}" for v in solver.vars)
    print(header)
    print("-" * len(header))

    for i in solver.vars:
        row = [f"{i:<5}"]
        for j in solver.vars:
            if i == j:
                cell = "{EQ}"
            else:
                cell = format_relations(solver.R[(i, j)])
            row.append(f"{cell:^26}")
        print(" | ".join(row))
    print()


regions = ("A", "B", "C")
solver = build_solver(regions)

print_matrix(solver, "Reseau initial: aucune contrainte connue")

Reseau initial: aucune contrainte connue
                  A              |             B              |             C             
------------------------------------------------------------------------------------------
A     |            {EQ}            | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI}
B     | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} |            {EQ}            | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI}
C     | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} |            {EQ}           



## 2. Ajout de contraintes connues

On suppose que:

- `A TPP B`: A est une partie propre tangentielle de B;
- `B TPP C`: B est une partie propre tangentielle de C.

Le solveur ajoute aussi automatiquement les relations inverses:

- `B TPPI A`;
- `C TPPI B`.

In [2]:
solver.add_constraint("A", "B", {"TPP"})
solver.add_constraint("B", "C", {"TPP"})

print_matrix(solver, "Apres ajout des contraintes A TPP B et B TPP C")

assert solver.R[("A", "B")] == {"TPP"}
assert solver.R[("B", "A")] == {"TPPI"}
assert solver.R[("B", "C")] == {"TPP"}
assert solver.R[("C", "B")] == {"TPPI"}

Apres ajout des contraintes A TPP B et B TPP C
                  A              |             B              |             C             
------------------------------------------------------------------------------------------
A     |            {EQ}            |           {TPP}            | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI}
B     |           {TPPI}           |            {EQ}            |           {TPP}           
C     | {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI} |           {TPPI}           |            {EQ}           



## 3. Composition attendue

La propagation entre `A` et `C` passe par `B`.

On applique la regle:

`R(A,C) <- R(A,C) intersection (R(A,B) compose R(B,C))`

Ici:

`TPP compose TPP = {TPP, NTPP}`

Donc `A` doit etre dans `C`, soit en touchant le bord (`TPP`), soit strictement a l'interieur (`NTPP`).

In [3]:
before = set(solver.R[("A", "C")])
composition = COMPOSE["TPP"]["TPP"]
after = before & composition

print("Domaine initial R(A,C):", format_relations(before))
print("Composition TPP o TPP:", format_relations(composition))
print("Domaine reduit attendu:", format_relations(after))

assert after == {"TPP", "NTPP"}

Domaine initial R(A,C): {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI}
Composition TPP o TPP: {TPP, NTPP}
Domaine reduit attendu: {TPP, NTPP}


## 4. Une etape de revision

La methode `revise(i, j, k)` applique exactement cette reduction pour le triplet `(i, j, k)`:

`R(i,k) <- R(i,k) intersection (R(i,j) compose R(j,k))`

In [4]:
changed = solver.revise("A", "B", "C")

print("Le domaine R(A,C) a change:", changed)
print_matrix(solver, "Apres revision du triplet A-B-C")

assert changed is True
assert solver.R[("A", "C")] == {"TPP", "NTPP"}
assert solver.R[("C", "A")] == {"TPPI", "NTPPI"}

Le domaine R(A,C) a change: True
Apres revision du triplet A-B-C
                  A              |             B              |             C             
------------------------------------------------------------------------------------------
A     |            {EQ}            |           {TPP}            |        {TPP, NTPP}        
B     |           {TPPI}           |            {EQ}            |           {TPP}           
C     |       {TPPI, NTPPI}        |           {TPPI}           |            {EQ}           



## 5. Propagation complete avec PC-2

PC-2 repete cette operation sur tous les triplets jusqu'a ce qu'aucune relation ne puisse plus etre retiree. On atteint alors un point fixe.

In [5]:
solver_pc2 = build_solver(regions)
solver_pc2.add_constraint("A", "B", {"TPP"})
solver_pc2.add_constraint("B", "C", {"TPP"})

solver_pc2.pc2()

print_matrix(solver_pc2, "Resultat apres fermeture PC-2")

assert solver_pc2.R[("A", "C")] == {"TPP", "NTPP"}
assert solver_pc2.R[("C", "A")] == {"TPPI", "NTPPI"}

Resultat apres fermeture PC-2
                  A              |             B              |             C             
------------------------------------------------------------------------------------------
A     |            {EQ}            |           {TPP}            |        {TPP, NTPP}        
B     |           {TPPI}           |            {EQ}            |           {TPP}           
C     |       {TPPI, NTPPI}        |           {TPPI}           |            {EQ}           



## 6. Trace des reductions

Pour mieux voir la propagation, on peut instrumenter une version simple de PC-2 qui enregistre les domaines modifies.

In [6]:
def trace_pc2(solver):
    """Execute PC-2 en enregistrant chaque reduction de domaine."""
    solver._normalize_network()
    solver._enforce_all_converses()

    steps = []
    changed = True
    iteration = 0

    while changed:
        iteration += 1
        changed = False

        for i in solver.vars:
            for j in solver.vars:
                if i == j:
                    continue

                for k in solver.vars:
                    if k in (i, j):
                        continue

                    before = set(solver.R[(i, k)])
                    revised = solver.revise(i, j, k)
                    after = set(solver.R[(i, k)])

                    if revised:
                        changed = True
                        steps.append({
                            "iteration": iteration,
                            "triplet": (i, j, k),
                            "before": before,
                            "after": after,
                        })

    return steps


trace_solver = build_solver(regions)
trace_solver.add_constraint("A", "B", {"TPP"})
trace_solver.add_constraint("B", "C", {"TPP"})

steps = trace_pc2(trace_solver)

for index, step in enumerate(steps, start=1):
    i, j, k = step["triplet"]
    print(f"Etape {index} | iteration {step['iteration']} | triplet {i}-{j}-{k}")
    print("  avant:", format_relations(step["before"]))
    print("  apres:", format_relations(step["after"]))

print()
print_matrix(trace_solver, "Reseau final apres trace PC-2")

assert steps
assert trace_solver.R[("A", "C")] == {"TPP", "NTPP"}

Etape 1 | iteration 1 | triplet A-B-C
  avant: {DC, EC, PO, EQ, TPP, NTPP, TPPI, NTPPI}
  apres: {TPP, NTPP}

Reseau final apres trace PC-2
                  A              |             B              |             C             
------------------------------------------------------------------------------------------
A     |            {EQ}            |           {TPP}            |        {TPP, NTPP}        
B     |           {TPPI}           |            {EQ}            |           {TPP}           
C     |       {TPPI, NTPPI}        |           {TPPI}           |            {EQ}           



## 7. Detection d'une incoherence indirecte

La propagation sert aussi a detecter les contradictions. Exemple:

- `A NTPP B`: A est strictement dans B;
- `B NTPP C`: B est strictement dans C;
- donc A doit etre strictement dans C;
- mais on impose `C DC A`: C est deconnecte de A.

Ces contraintes ne peuvent pas toutes etre vraies en meme temps.

In [7]:
bad_solver = build_solver(regions)
bad_solver.add_constraint("A", "B", {"NTPP"})
bad_solver.add_constraint("B", "C", {"NTPP"})
bad_solver.add_constraint("C", "A", {"DC"})

try:
    bad_solver.pc2()
    print("Aucune incoherence detectee")
except ValueError as error:
    print("Incoherence detectee:", error)

Incoherence detectee: Inconsistency detected between A and C


## 8. A retenir

- Une contrainte RCC8 est un ensemble de relations possibles entre deux regions.
- Ajouter une relation directe reduit le domaine de la paire concernee et de sa paire inverse.
- La table de composition permet de deduire des relations indirectes.
- PC-2 applique ces deductions sur tous les triplets jusqu'au point fixe.
- Si un domaine devient vide, le reseau spatial est incoherent.